# 📊 Bond Payment Data
<br>

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Three-Chapter-Three/blob/master/colab/Colab_Bootstrapping_Zero_Prices.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Three-Chapter-Three/master?urlpath=lab/tree/notebooks/Bootstrapping Zero Prices.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Three-Chapter-Three/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>
</div>

📅 A bond's accrued interest is calculated based on scheduled payment amounts and dates, but its present value requires the actual amounts and dates. Although the payment amounts are fixed, the actual payment dates often deviate from the scheduled ones.

🗓️ Actual payment dates correspond to settlement dates, while scheduled payments are determined relative to the bond's maturity date. Since settlement days exclude weekends, holidays, and Good Friday, any payments scheduled for these days are advanced to the next available settlement day.
The `bond_payment_data` function in the notebook is used to determine both payment dates and amounts. This function relies on `adjust_bond_pay_dates`, which exemplifies the "Iceberg Principle." It utilizes the pandas_market_calendars library, a powerful tool providing valid trade dates for over fifty exchanges (including NYSE, LSE, and EUREX) and bond calendars for the U.S., U.K., and Japan.

🇺🇸 For this notebook, the U.S. calendar, `SIFMAUS`, is employed. The calendar is based on FED bank holidays and incorrectly treats Good Friday as a settlement date. To correct this the Pandas module, `tseries.holidays`and the rule `GoodFriday` are imported from the Pandas `tseries` module. This allows for a modification of the SIFMAUS-created calendar to accurately account for Good Friday, resulting in the correct set of settlement dates for U.S. bonds. The NumPy function `busday_offset` is used to account for non setlement dates, including Good Friday.

📓 The "Iceberg Principle" is further demonstrated in the companion notebook, *Bootstrapping Zero Prices*. That notebook leverages the `bond_payment_data`, `accrued_interest`, `FEDInvest`, and `clean_FEDInvest` functions to calculate zero prices from a sample of coupon bonds.


## Preparing the notebook

<details style="border: 1px solid #b8daff; border-radius: 8px; padding: 15px; background-color: #f8fbff; margin: 20px 0;">
  <summary style="cursor: pointer; font-weight: bold; font-size: 1.1em; color: #004085; font-family: sans-serif; list-style-position: inside;">
    <span style="margin-right: 8px;">🛠️</span> Notebook Setup: Why the "Try/Except" Imports?
  </summary>
  <div style="margin-top: 15px; padding-top: 15px; border-top: 1px solid #b8daff; color: #333333; font-family: sans-serif; font-size: 0.95em;">
    <b>The Goal:</b><br>
    To ensure this notebook runs perfectly whether you are using <b>Google Colab</b>, a local <b>Jupyter instance</b>, or a remote server without you having to manually install software.<br><br>
    <b>Key Concepts in this Section:</b>
    <ul style="line-height: 1.6; margin-bottom: 0;">
      <li><b>Standard Libraries:</b> Modules like <code>os</code>, <code>sys</code>, and <code>datetime</code> come "in the box" with Python. We use them for system tasks and date math.</li>
      <li><b>External Libraries:</b> NumPy and Pandas are the "heavy hitters" for data. They aren't always installed by default.</li>
      <li><b>The <code>try/except</code> Logic:</b> This is a safety net.
        <ol style="margin-top: 4px; margin-bottom: 4px; padding-left: 20px;">
          <li>We <b>try</b> to import the library.</li>
          <li>If it fails (because it's not installed), the <b>except</b> block triggers a <code>!pip install</code> to download it automatically.</li>
        </ol>
      </li>
      <li><b>Aliasing (<code>as np</code>):</b> We rename <code>numpy</code> to <code>np</code> to save keystrokes. In professional finance code, <code>np</code> and <code>pd</code> are the universal shorthand.</li>
    </ul>
  </div>
</details>

### Importing libraries, modules, And functions

Modules that are included in the standard Python library are imported. When necessary, other modules or libraries are installed before they are imported. (see [Control Statements](https://patrickjhess.github.io/Introduction-To-Python-For-Financial-Python/Control_Statements.html#the-try-and-except)).

```
import os
import sys
import requests
from datetime import date, datetime
from types import ModuleType

try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd


try:
    import numpy as np
except:
    !pip -q install numpy
    import numpy as np

try:
  import pandas_market_calendars as mcal
except:
  !pip -q install pandas_market_calendars
  import pandas_market_calendars as mcal

from pandas.tseries.holiday import GoodFriday
```

In [ ]:
import os
import sys
import requests
from datetime import date, datetime
from types import ModuleType
try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd

try:
    import numpy as np
except:
    !pip -q install numpy
    import numpy as np

try:
  import pandas_market_calendars as mcal
except:
  !pip -q install pandas_market_calendars
  import pandas_market_calendars as mcal

from pandas.tseries.holiday import GoodFriday

## 📦 Getting `bond_pay_data` from the custom module

<details>
<summary><b style="font-size:1.2em; color: #1976d2; cursor: pointer;">👍 Important: Cloud-Loading: How In-Memory Modules Work</b></summary>
<br>
<p><b>The Logic:</b><br>
Usually, Python looks for modules as <code>.py</code> files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.</p>

<p><b>The Workflow:</b></p>
<ol>
<li><b>Fetch:</b> <code>requests.get(url)</code> grabs the raw text of your Python script from Dropbox.</li>
<li><b>Instantiate:</b> <code>ModuleType(module_name)</code> creates an empty "container" in your computer's RAM.</li>
<li><b>Execute:</b> <code>exec(code, module.__dict__)</code> runs that text inside the container, turning text into live functions.</li>
<li><b>Register:</b> By adding it to <code>sys.modules</code>, we tell Python: <em>"If I try to import this later, don't look on the disk—look right here in the memory."</em></li>
</ol>

<p><b>Why do this?</b><br>
It makes your notebooks <b>100% portable</b>. A user can open this in a brand-new environment, and as long as they have an internet connection, all your custom financial functions will "just work."</p>
</details>

### Adding a custom module and importing functions


Now that we’ve imported our main modules and libraries, we’ll access a custom module for our more specific functions. In the code below, the custom module `module_basic_concepts_fixed_income` contains functions utilized by this notebook and others in the volume Basic Concepts of Fixed Income.

We access this module from Dropbox using `requests.get()`. This approach allows the notebook to remain "portable"—it fetches the necessary tools directly from the cloud without requiring you to manage local files.

The module is instantiated with `ModuleType` (imported earlier). Once created, the module becomes accessible in the Notebook’s memory, though it is not saved to your hard drive. The `exec()` function then executes the Python code returned by the URL and assigns it to `sys.module`, making the functions ready for use.

Finally, we import the functions `bond_pay_data`.


```
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.text, module.__dict__)
    sys.modules[module_name] = module
    # Now we can import from our in-memory module
    from module_basic_concepts_fixed_income import (bond_pay_data)
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")
```

Our now-complete code is shown in the cell below.

In [ ]:
# Define the URL of the Python module to be downloaded from Dropbox.
# The 'dl=1' parameter in the URL forces a direct download of the file content.
url= 'https://www.dropbox.com/scl/fi/4y5hjxlfphh1ngvbgo77q/\
module_-basic_concepts_fixed_income.py?rlkey=6oxi7mgka42veaat79hcv8boz&st=87sztshr&dl=1'
module_name='basic_concepts_fixed_income'
# Send an HTTP GET request to the URL and store the server's response.
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.text, module.__dict__)
    sys.modules[module_name] = module
    # Now we can import from our in-memory module
    from basic_concepts_fixed_income import (bond_pay_data)
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")
    # Now we can import from our in-memory module

## 📓 This notebook explores two new features: the 📅 pandas_market_calendars library and the ↪️ busday_offset NumPy function. To help you experiment with these concepts, two code snippets are provided below.
* 🎄 Snippet 1: The first snippet uses the pandas_market_calendars library to find non-settlement days between December 21, 2025, and January 1, 2026. Christmas and New Year's are successfully detected, while Good Friday naturally does not occur within this date range.

* ⚙️ Snippet 2: The second snippet utilizes busday_offset to assign a valid settlement day to these holidays, as well as to regular weekends. This snippet assumes that the Datetime indexes dates, fed_holidays_idx, and Good_Friday_idx were previously created by the first snippet.



In [ ]:
# @title ✂️ Click show code for `pandas_market_calendars` snippets or execute
try:
  import pandas_market_calendars as mcal
except:
  !pip -q install pandas_market_calendars
  import pandas_market_calendars as mcal
from pandas.tseries.holiday import GoodFriday
import pandas as pd

# create datetime index
start = pd.Timestamp(2025,12,21)
end = pd.Timestamp(2026,1,10)
dates = pd.date_range(start=start, end=end)

# use the library to create the object
fed_cal = mcal.get_calendar('SIFMAUS')
fed_holidays = fed_cal.holidays().holidays

# get Good Fridays
Good_Fridays = GoodFriday.dates(start,end)

# create Pandas Datetime indexes
fed_holiday_idx = pd.DatetimeIndex(fed_holidays)
Good_Friday_idx = pd.DatetimeIndex(Good_Fridays)

# create filter for fed holidays to limit number
fed_holidays_start_end = (fed_holiday_idx >= start) & (fed_holiday_idx <= end)

# display the results for Panda Datetime indexes
display(fed_holiday_idx[fed_holidays_start_end])
display(Good_Friday_idx)

DatetimeIndex(['2025-12-25', '2026-01-01'], dtype='datetime64[s]', freq=None)

DatetimeIndex([], dtype='datetime64[us]', freq=None)

In [ ]:
# @title ✂️ Click show code for `NumPy busday_offset` snippets or execute
# Numpy dates must be datetime64
# combine fed_and Good Friday Datetime indexes
combined_holidays_idx=fed_holiday_idx.union(Good_Friday_idx)
numpy_holidays = combined_holidays_idx.values.astype('datetime64[D]')

# Use NumPy for fully vectorized date math
actual_payment_dates = np.busday_offset(
    dates.values.astype('datetime64[D]'),
    offsets=0,
    roll='forward',
    holidays=numpy_holidays
)

# convert actual dates to datetime.date
settlement_dates = pd.to_datetime(actual_payment_dates).date

# dates that need adjusting
adjusted_dates = [{actual, settlement}
                  for actual, settlement in zip(dates.date, settlement_dates)
                  if actual != settlement]
display(adjusted_dates)

[{datetime.date(2025, 12, 21), datetime.date(2025, 12, 22)},
 {datetime.date(2025, 12, 25), datetime.date(2025, 12, 26)},
 {datetime.date(2025, 12, 27), datetime.date(2025, 12, 29)},
 {datetime.date(2025, 12, 28), datetime.date(2025, 12, 29)},
 {datetime.date(2026, 1, 1), datetime.date(2026, 1, 2)},
 {datetime.date(2026, 1, 3), datetime.date(2026, 1, 5)},
 {datetime.date(2026, 1, 4), datetime.date(2026, 1, 5)},
 {datetime.date(2026, 1, 10), datetime.date(2026, 1, 12)}]

## **Getting the actual payment dates and amounts**

🧾 Getting the actual payment dates and amounts
The bond_pay_data function is used to calculate both the exact payment dates and the corresponding payment amounts for a given bond.

* **📥 Inputs & Defaults**: The function requires the bond's maturity date and coupon rate. By default, the settlement date is set to the current day, and the payment frequency defaults to 2 (semi-annual payments). However, both of these can be manually specified if needed.

* **🛠️ Under the Hood**: To perform these calculations, bond_pay_data relies on two essential helper functions: scheduled_pay_dates (introduced in Chapter Two) and adjust_bond_pay_dates (built in this notebook).

* **📤 Outputs**: The function returns an array containing the calculated dates and payment amounts.

📈 These finalized dates and cash flows are exactly what we need for the next notebook, where they will be used to bootstrap zero prices from coupon bonds.
<details>
<summary> 🔍<b>Click to see the function</b></summary>

<br>

```python
def bond_pay_data(maturity, coupon, settlement=None, freq=2):
    '''
    Function calculates payment Dates And Amounts.
    maturity is a datetime object and coupon is a real number.
    Required arguments are maturity and annual coupon.
    If provided, the value of settlement is a datetime object;
    otherwise defaults to date.today()
    freq defaults to semi-annual but accepts freq equal
    to 1 for annual, 2 for semi-annal, 4 for quarterly, and 12 for monthly.
    The function assumes a par value of 100.
    Returns Numpy arrays of dates and amounts.

    Raises:
        TypeError: If maturity or settlement are not datetime objects.
        ValueError: If inputs are not logically valid (e.g., negative coupon,
                    maturity before settlement).
    '''
    from datetime import datetime, date
    from dateutil.relativedelta import relativedelta
    import pandas as pd
    import numpy as np
    from IPython.display import display, Markdown as md

    # Validate the data - maturity, coupon, settlement, freq
    def validate_date(datetime_object):
        # check for datetime or date
        if not isinstance(datetime_object, (datetime, date)):
            raise TypeError("Input must be a datetime or date object.")
        # convert datetime to date
        if isinstance(datetime_object, datetime):
            datetime_object = datetime_object.date()
        return datetime_object

    # maturity
    maturity = validate_date(maturity)

    # settlement
    if settlement is None:
        settlement = date.today()
    else:
        settlement = validate_date(settlement)

    # coupon
    try:
        coupon = float(coupon)
        if coupon < 0:
            raise ValueError("coupon rate cannot be negative.")
    except (ValueError, TypeError):
        raise ValueError("coupon must be a valid number.")

    # freq
    if int(freq) not in [1, 2, 4, 12]:
        display(md(f"### ⚠️ your assigned freq {freq} it must be (1, 2, 4, or 12)\n ### semi-annual assumed (2)."))
        freq = int(2)

    # check maturity greater than settlement
    if maturity <= settlement:
        raise ValueError("maturity must be greater than the settlement date")

    if coupon == 0:
        # Adjust maturity for non-settlement day and return date and face value
        adjust_maturity = adjust_bond_pay_dates(maturity)
        return np.array([adjust_maturity['Settlement'].dt.date]), np.array([100.0])

    # get scheduled payment dates from helper function scheduled_pay_dates
    scheduled_dates = scheduled_pay_dates(maturity, settlement, freq)

    # Pandas DataFrame Settlement desired column
    both_dates = adjust_bond_pay_dates(scheduled_dates)
    pay_dates=np.array(both_dates['Settlement'].dt.date)
    # calculate payments
    # coupon divided by freq at each date
    pay = np.full(len(pay_dates), coupon / freq)

    # Add principal payment as last cash payment
    pay[-1] += 100

    return pay_dates,pay

    </details>


<div style="padding: 15px; border: 1px solid #e0e0e0; border-left: 5px solid #ffc107; background-color: #fffde7; border-radius: 4px; margin-bottom: 15px;">
  <b>✍️ Application:</b> Calculate the payment dates and amounts for two bonds that mature on August 31, 2035. One bond has a zero coupon and the other an annual coupon of 4.<br><br>
  <b>The Challenge:</b> Use the function <code>bond_pay_data</code>.<br>
  <b>Recall:</b> The 'Iceberg Principle' of functions that depend upon other functions that are imported.
</div>

<details>
<summary>🛟 <b>Need hints or a solution?</b></summary>

<br>

<details style="margin-left: 20px;">
<summary>💡 <b>Hints</b></summary>
<ul>
  <li><b>Imports:</b> Refer to the "Preparing notebook" section of this notebook</li>
  <li><b>Create the maturity and settlement dates:</b> <code>date(year, month, day)</code></li>
</ul>
</details>

<br>

<details style="margin-left: 20px;">
<summary>✅ <b>Example Of Solution</b></summary>

<br>

```python
# set the maturity and settlement date
maturity = date(2035, 8, 31)
settlement = date(2026, 4, 21)

# iterate through the coupons and display results
for coupon in [4, 0]:
    display(bond_pay_data(maturity, coupon, settlement=settlement, freq=2))